# Sentiment Analysis - Minecraft Reviews (TensorFlow)
**Nama**: Asep Haryana Saputra

## Project Overview
**Objective**: Build a high-accuracy (>92%) sentiment analysis model.
**Dataset**: 40,000 Minecraft reviews.
**Framework**: TensorFlow / Keras.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import tensorflow as tf
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
try:
    from tensorflow.keras.layers import TextVectorization, SpatialDropout1D, BatchNormalization, Conv1D, GlobalMaxPooling1D
except ImportError:
    from tensorflow.keras.layers.experimental.preprocessing import TextVectorization
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, GRU, Dropout, GlobalAveragePooling1D, Input, SpatialDropout1D, BatchNormalization, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.models import Sequential, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

# Optimize GPU Memory
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


TensorFlow Version: 2.19.0


In [2]:
# Download and Extract GloVe Embeddings (if not present)
import os
import urllib.request
import zipfile
import ssl

GLOVE_URL = "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip"
ZIP_FILE = "glove.6B.zip"
TXT_FILE = "glove.6B.100d.txt"

if not os.path.exists(TXT_FILE):
    print(f"Downloading {ZIP_FILE}...")
    # SSL context to avoid certificate errors
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    try:
        with urllib.request.urlopen(GLOVE_URL, context=ctx) as response, open(ZIP_FILE, 'wb') as out_file:
            chunk_size = 1024 * 1024 # 1MB
            while True:
                chunk = response.read(chunk_size)
                if not chunk: break
                out_file.write(chunk)

        print("Extracting...")
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract(TXT_FILE)

        # Cleanup
        if os.path.exists(ZIP_FILE):
            os.remove(ZIP_FILE)
        print("GloVe embeddings ready.")
    except Exception as e:
        print(f"Error downloading GloVe: {e}")
else:
    print(f"{TXT_FILE} already exists.")



Extracting...
GloVe embeddings ready.


In [3]:
# Callbacks Definition
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

def get_callbacks(model_name):
    checkpoint = ModelCheckpoint(
        f'{model_name}_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        min_delta=0.001,
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
    return [checkpoint, early_stopping, reduce_lr]

## 2. Data Loading & Preprocessing
**Labeling Strategy**:
- 1-2 Stars -> **Negative (0)**
- 3 Stars -> **Neutral (1)**
- 4-5 Stars -> **Positive (2)**

In [4]:
# Load Data
df = pd.read_csv('dataset_raw.csv')
df.dropna(subset=['content'], inplace=True)

# Labeling
def map_sentiment(score):
    if score <= 2: return 0
    elif score == 3: return 1
    else: return 2

df['label'] = df['score'].apply(map_sentiment)

# Clean Text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text) # Keep numbers & punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_content'] = df['content'].apply(clean_text)
print(df['label'].value_counts())

label
2    31359
0     6988
1     1650
Name: count, dtype: int64


In [5]:
# 3. Data Filtering & Splitting
# Remove extremely short reviews (noise)
if 'content' in df.columns:
    df['word_count'] = df['content'].apply(lambda x: len(str(x).split()))
    df_clean = df[df['word_count'] >= 3].copy()
    print(f"Original shape: {df.shape}, Filtered shape: {df_clean.shape}")

    X = df_clean['content'].values
    y = df_clean['label'].values

    # Scheme 1 & 3: 80/20 Split
    X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Scheme 2: 90/10 Split
    X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

    print(f"Scheme 1/3 (80/20) - Train: {X_train_80.shape}, Test: {X_test_80.shape}")
    print(f"Scheme 2 (90/10)   - Train: {X_train_90.shape}, Test: {X_test_90.shape}")
else:
    print("Error: 'content' column not found yet. Ensure Data Loading cell is run first.")

Original shape: (39997, 14), Filtered shape: (32761, 14)
Scheme 1/3 (80/20) - Train: (26208,), Test: (6553,)
Scheme 2 (90/10)   - Train: (29484,), Test: (3277,)


## 3. Tokenization & Splitting

In [6]:
# Tokenization for Scheme 1 & 2 (Embedding Models)
VOCAB_SIZE = 10000
MAX_LENGTH = 100
EMBEDDING_DIM = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

# Prepare Sequences for Scheme 1 (80/20)
seq_train_80 = tokenizer.texts_to_sequences(X_train_80)
seq_test_80 = tokenizer.texts_to_sequences(X_test_80)
pad_train_80 = pad_sequences(seq_train_80, maxlen=MAX_LENGTH, padding='post', truncating='post')
pad_test_80 = pad_sequences(seq_test_80, maxlen=MAX_LENGTH, padding='post', truncating='post')

# Prepare Sequences for Scheme 2 (90/10)
seq_train_90 = tokenizer.texts_to_sequences(X_train_90)
seq_test_90 = tokenizer.texts_to_sequences(X_test_90)
pad_train_90 = pad_sequences(seq_train_90, maxlen=MAX_LENGTH, padding='post', truncating='post')
pad_test_90 = pad_sequences(seq_test_90, maxlen=MAX_LENGTH, padding='post', truncating='post')

## 3.1 Load Pre-trained GloVe Embeddings


In [7]:
# Load GloVe Embeddings
embeddings_index = {}
GLOVE_PATH = 'glove.6B.100d.txt'
try:
    with open(GLOVE_PATH, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    print(f'Found {len(embeddings_index)} word vectors.')
except FileNotFoundError:
    print("GloVe file not found. Please ensure glove.6B.100d.txt is in the directory.")

# Create Embedding Matrix
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))
hits = 0
misses = 0

for word, i in tokenizer.word_index.items():
    if i < VOCAB_SIZE:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
            hits += 1
        else:
            misses += 1

print(f"Converted {hits} words ({hits+misses} total)")

Found 400001 word vectors.
Converted 7221 words (9999 total)


In [8]:
# Force CPU for TextVectorization to avoid GPU OOM
with tf.device('/CPU:0'):
    # Preprocessing for Scheme 3 (TF-IDF)
    vectorize_layer = TextVectorization(
        max_tokens=VOCAB_SIZE,
        output_mode='tf-idf',
        ngrams=2 # improved features
    )
    # Adapt on 80% train data
    vectorize_layer.adapt(X_train_80)

    tfidf_train_80 = vectorize_layer(X_train_80)
    tfidf_test_80 = vectorize_layer(X_test_80)


## 3.1 Load Pre-trained Embeddings


In [9]:
# Load GloVe Embeddings
embeddings_index = {}
GLOVE_PATH = 'glove.6B.100d.txt'

print("Loading GloVe vectors...")
try:
    with open(GLOVE_PATH, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    print(f'Found {len(embeddings_index)} word vectors.')
except FileNotFoundError:
    print("GloVe file not found. Run the download cell above.")

# Create Embedding Matrix
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))
hits = 0
misses = 0

for word, i in tokenizer.word_index.items():
    if i < VOCAB_SIZE:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
            hits += 1
        else:
            misses += 1

print(f"Converted {hits} words ({hits+misses} total)")

Loading GloVe vectors...
Found 400001 word vectors.
Converted 7221 words (9999 total)


## 4. Experiment 1: Bi-LSTM Model

In [10]:
# Scheme 1 (Baseline): Bi-LSTM + Embedding + 80/20 Split
tf.random.set_seed(42)
model1 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme1_BiLSTM')

model1.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model1.summary()

history1 = model1.fit(
    pad_train_80, y_train_80,
    epochs=50,
    validation_data=(pad_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme1_bilstm')
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "Scheme1_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7572 - loss: 0.6977
Epoch 1: val_accuracy improved from -inf to 0.84389, saving model to scheme1_bilstm_best.keras
819/819 ━━━━━━━━━━━━━━━━━━━━ 31s 26ms/step - accuracy: 0.7573 - loss: 0.6976 - val_accuracy: 0.8439 - val_loss: 0.4772 - learning_rate: 0.0010
Epoch 2/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8596 - loss: 0.4307
Epoch 2: val_accuracy improved from 0.84389 to 0.85213, saving model to scheme1_bilstm_best.keras
819/819 ━━━━━━━━━━━━━━━━━━━━ 35s 25ms/step - accuracy: 0.8596 - loss: 0.4307 - val_accuracy: 0.8521 - val_loss: 0.4551 - learning_rate: 0.0010
Epoch 3/50
817/819 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8807 - loss: 0.3586
Epoch 3: val_accuracy did not improve from 0.85213
819/819 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.8807 - loss: 0.3586 - val_accuracy: 0.8468 - val_loss: 0.4818 - learning_rate: 0.0010
Epoch 4/50
818/819 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0

## 5. Experiment 2: GRU Model

In [ ]:
# Scheme 2 (Data Variation): GRU + Embedding + 90/10 Split
tf.random.set_seed(42)
model2 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    GRU(128, return_sequences=True),
    Dropout(0.3),
    GRU(64),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme2_GRU_90_10')

model2.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model2.summary()

history2 = model2.fit(
    pad_train_90, y_train_90,
    epochs=50,
    validation_data=(pad_test_90, y_test_90),
    batch_size=32,
    callbacks=get_callbacks('scheme2_gru')
)

Model: "Scheme2_GRU_90_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
922/922 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6838 - loss: 0.8548
Epoch 1: val_accuracy improved from -inf to 0.74825, saving model to scheme2_gru_best.keras
922/922 ━━━━━━━━━━━━━━━━━━━━ 17s 16ms/step - accuracy: 0.6839 - loss: 0.8547 - val_accuracy: 0.7482 - val_loss: 0.7188 - learning_rate: 0.0010
Epoch 2/50
919/922 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7512 - loss: 0.7094
Epoch 2: val_accuracy did not improve from 0.74825
922/922 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.7512 - loss: 0.7094 - val_accuracy: 0.7482 - val_loss: 0.6975 - learning_rate: 0.0010
Epoch 3/50
 21/922 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.7648 - loss: 0.6622

## 6. Experiment 3: Simple NN (Baseline)

In [ ]:
# Scheme 3 (Feature Variation): Dense + TF-IDF + 80/20 Split
tf.random.set_seed(42)
model3 = Sequential([
    Input(shape=(VOCAB_SIZE,)), # TF-IDF input
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(3, activation='softmax')
], name='Scheme3_Dense_TFIDF')

model3.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model3.summary()

history3 = model3.fit(
    tfidf_train_80, y_train_80,
    epochs=50,
    validation_data=(tfidf_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme3_tfidf')
)

## 7. Experiment 4: Bi-LSTM + GloVe Embeddings (Transfer Learning)


In [ ]:
# Scheme 4: Bi-LSTM with Frozen GloVe Embeddings
tf.random.set_seed(42)
model4 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM,
              embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
              trainable=False, # Freeze embeddings first
              input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme4_GloVe_BiLSTM')

model4.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model4.summary()

history4 = model4.fit(
    pad_train_80, y_train_80,
    epochs=50,
    validation_data=(pad_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme4_glove')
)


## 8. Experiment 5: Conv1D (N-Gram Learning)


In [ ]:
# Scheme 5: Conv1D for N-Gram feature extraction
# Conv1D filters act as N-gram detectors (e.g. kernel_size=2 sees 2 words)
tf.random.set_seed(42)
model5 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Conv1D(128, 5, activation='relu'), # Detects 5-grams
    GlobalMaxPooling1D(), # Captures most important features
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme5_Conv1D_NGram')

model5.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model5.summary()

history5 = model5.fit(
    pad_train_80, y_train_80,
    epochs=50,
    validation_data=(pad_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme5_conv1d')
)


## 7. Evaluation & Inference

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    print(f"--- Evaluation: {name} ---")
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test Accuracy: {acc*100:.2f}%")
    y_pred = np.argmax(model.predict(X_test), axis=1)
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))
    print("-"*30 + "\n")

# Evaluate all 3 schemes
evaluate_model(model1, pad_test_80, y_test_80, "Scheme 1 (Bi-LSTM, 80/20)")
evaluate_model(model2, pad_test_90, y_test_90, "Scheme 2 (GRU, 90/10)")
evaluate_model(model3, tfidf_test_80, y_test_80, "Scheme 3 (TF-IDF, 80/20)")
evaluate_model(model4, pad_test_80, y_test_80, "Scheme 4 (GloVe)")
evaluate_model(model5, pad_test_80, y_test_80, "Scheme 5 (Conv1D)")

# Compare All Validation Accuracies
plt.figure(figsize=(12, 6))
plt.plot(history1.history['val_accuracy'], label='Scheme 1 (Bi-LSTM)')
plt.plot(history2.history['val_accuracy'], label='Scheme 2 (GRU)')
plt.plot(history3.history['val_accuracy'], label='Scheme 3 (TF-IDF)')
plt.plot(history4.history['val_accuracy'], label='Scheme 4 (GloVe)')
plt.plot(history5.history['val_accuracy'], label='Scheme 5 (Conv1D)')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


In [ ]:
# Comparison of All Schemes
plt.figure(figsize=(12, 6))
plt.plot(history1.history['val_accuracy'], label='Scheme 1 (Bi-LSTM)')
plt.plot(history2.history['val_accuracy'], label='Scheme 2 (GRU)')
plt.plot(history3.history['val_accuracy'], label='Scheme 3 (TF-IDF)')
try:
    plt.plot(history4.history['val_accuracy'], label='Scheme 4 (GloVe)')
    plt.plot(history5.history['val_accuracy'], label='Scheme 5 (Conv1D)')
except:
    pass
plt.title('Validation Accuracy: Schemes 1-5')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()



In [ ]:
# Inference Function
def predict_sentiment(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding=PADDING_TYPE, truncating=TRUNC_TYPE)
    pred = model1.predict(padded)
    label_idx = np.argmax(pred)

    mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return mapping[label_idx]

# Test
print(predict_sentiment("This game is super fun!"))
print(predict_sentiment("Too many bugs, unplayable."))